# Pregnancy Outcomes Extraction
## Defining Pregnancy Outcomes for Prediction Models

Extract and define pregnancy outcome variables (dependent variables) for predictive models

**Outcomes extracted:**
- Pre-eclampsia (concept_id: 439393)
- Gestational Diabetes (concept_id: 4024659)
- Stillbirth (concept_id: 443213)
- Premature birth (concept_id: 4086393)
- Underweight (concept_id: 4145947)
- Eclampsia (concept_id: 443700)
- Gestational Hypertension (concept_id: 4167493)
- Placental Abruption (concept_id: 198488)

**Time-specific indicators:** Creates `_12` suffix versions for 12-week prediction models (outcomes must occur after 12 weeks from LMP)

File is commented out because it builds a dataset in the All of Us open research access dataset. If you have an account, you can uncomment the code to run and duplicate our cohort. 


# Setup

In [ ]:
'''
from IPython.display import display, HTML
import pandas as pd
import numpy as np, scipy
import matplotlib, matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import date
import seaborn as sns
import os
import subprocess
import pickle
from sklearn.model_selection import train_test_split
'''

In [ ]:
'''
version = %env WORKSPACE_CDR
version
'''

# Load Base Cohort

In [ ]:
'''
pregnancy_episodes_df = pd.read_pickle("./pregnancy_episodes_date.pkl")
print(pregnancy_episodes_df.head())

# Get unique person IDs for queries
person_ids = list(set(pregnancy_episodes_df['person_id']))

# Define outcomes to extract
outcomes = ['Pre-eclampsia', 'Gestational Diabetes', 'Stillbirth', 'Premature', 
            'Underweight', 'Eclampsia', 'Gestational_Hypertension', 'Placental_Abruption']

print(f"\nTotal pregnancy episodes: {len(pregnancy_episodes_df):,}")
print(f"Unique individuals: {len(person_ids):,}")
print(f"\nOutcomes to extract: {len(outcomes)}")
'''

# Helper Functions

In [ ]:
'''
def custom_sql(person_ids, sql_command, num_splits = 5): 
    """Split large queries into chunks to avoid BigQuery timeouts"""
    CDR = os.environ['WORKSPACE_CDR']

    div_index = len(person_ids) // num_splits

    for i in range(num_splits + 1): 
      
        ids_string = ', '.join(str(e) for e in person_ids[i*div_index:(i + 1)*div_index])
        if len(ids_string) == 0:
            continue
        
        df_temp = pd.read_gbq(sql_command.format(CDR, ids_string),
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            dialect = "standard", progress_bar_type="tqdm_notebook")

        if i == 0: 
            df = df_temp 
        else: 
            df = pd.concat([df, df_temp])
    return df

def generate(df, outcome, name, early):
    """
    Create binary outcome indicator for pregnancies
    
    Parameters:
    - df: pregnancy episodes dataframe
    - outcome: condition records for this outcome
    - name: column name for outcome
    - early: 'early' to require diagnosis after 12w, otherwise any time after LMP
    """
    outcome['start_date'] = pd.to_datetime(outcome['CONDITION_START_DATETIME'])

    df[name] = 0
    for i, row in df.iterrows():
        censor_date = row['censor_date']
        delivery_date = row['delivery_date']
        person_id = row['person_id']

        outcome_filter = outcome[outcome['PERSON_ID'] == person_id]

        for start_date in outcome_filter['start_date']:
            # For 12-week models: diagnosis must occur after 12 weeks from LMP
            if (early == 'early') and (censor_date + pd.DateOffset(weeks=12) < start_date) and (start_date <= (delivery_date + pd.DateOffset(weeks = 4))):
                df.at[i, name] = 1
                break
            # For all-pregnancy models: diagnosis any time after LMP
            elif (censor_date < start_date) and (start_date <= (delivery_date + pd.DateOffset(weeks = 4))):
                df.at[i, name] = 1
                break
    return(df)

print("Helper functions defined")
'''

# Extracting Outcomes

## Extract Pre-eclampsia

**Concept ID:** 439393 (Pre-eclampsia and descendants)

In [ ]:
'''
preeclampsia_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    439393
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying pre-eclampsia...")
df_preeclampsia = custom_sql(person_ids, preeclampsia_sql, num_splits = 3)

# Generate outcome indicators
df = generate(pregnancy_episodes_df, df_preeclampsia, outcomes[0], 'not')
df = generate(pregnancy_episodes_df, df_preeclampsia, str(outcomes[0]) + "_12", 'early')

print(f"Pre-eclampsia cases: {len(df[df[outcomes[0]] == 1]):,}")
'''

## Extract Gestational Diabetes

**Concept ID:** 4024659 (Gestational diabetes mellitus)

In [ ]:
'''
gestdb_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    4024659
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying gestational diabetes...")
df_gestdb = custom_sql(person_ids, gestdb_sql, num_splits = 3)

df = generate(df, df_gestdb, outcomes[1], 'not')
print(f"Gestational Diabetes cases: {len(df[df[outcomes[1]] == 1]):,}")

df = generate(df, df_gestdb, str(outcomes[1]) + "_12", 'early')
'''

## Extract Stillbirth

**Concept ID:** 443213 (Stillbirth and descendants)

In [ ]:
'''
still_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    443213
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying stillbirth...")
df_still = custom_sql(person_ids, still_sql, num_splits = 3)

print(f"Stillbirth records shape: {df_still.shape}")

df = generate(df, df_still, outcomes[2], 'not')
print(f"Stillbirth cases: {len(df[df[outcomes[2]] == 1]):,}")

df = generate(df, df_gestdb, str(outcomes[2]) + "_24", 'early')
'''

## Extract Premature Birth

**Concept ID:** 4086393 (Premature birth and descendants)

In [ ]:
'''
pre_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    4086393
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying premature birth...")
df_premature = custom_sql(person_ids, pre_sql, num_splits = 3)

df = generate(df, df_premature, outcomes[3], 'not')
print(f"Premature birth cases: {len(df[df[outcomes[3]] == 1]):,}")
'''

## Extract Underweight (SGA)

**Concept ID:** 4145947 (Low birth weight)

In [ ]:
'''
low_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    4145947
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

df_low = custom_sql(person_ids, low_sql, num_splits = 3)
'''

In [ ]:
'''
df = generate(df, df_low, outcomes[4], None)
print(len(df[df[outcomes[4]] == 1]))
'''

## Extract Eclampsia

**Concept ID:** 443700 (Eclampsia)

In [ ]:
'''
eclampsia_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    443700
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying eclampsia...")
df_eclampsia = custom_sql(person_ids, eclampsia_sql, num_splits = 3)

df = generate(df, df_eclampsia, outcomes[5], 'not')
print(f"Eclampsia cases: {len(df[df[outcomes[5]] == 1]):,}")
'''

## Extract Gestational Hypertension

**Concept ID:** 4167493 (Gestational hypertension)

In [ ]:
'''
gest_htn_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    4167493
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying gestational hypertension...")
df_gest_htn = custom_sql(person_ids, gest_htn_sql, num_splits = 3)

df = generate(df, df_gest_htn, outcomes[6], 'not')
df = generate(df, df_gest_htn, str(outcomes[6]) + "_12", 'early')

print(f"Gestational Hypertension cases: {len(df[df[outcomes[6]] == 1]):,}")
'''

## Extract Placental Abruption

**Concept ID:** 198488 (Placental abruption)

In [ ]:
'''
abrupt_sql = """
SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.condition_occurrence` c_occurrence 
    WHERE 
        person_id IN ({1})
        AND condition_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `{0}.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                            where
                                domain_id = 'CONDITION' 
                                and is_standard = 1 
                                and concept_id in (
                                    198488
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[condition_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) c_occurrence
LEFT JOIN
    `{0}.concept` c_standard_concept 
        on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID

"""

print("Querying placental abruption...")
df_abrupt = custom_sql(person_ids, abrupt_sql, num_splits = 3)

df = generate(df, df_abrupt, outcomes[7], 'not')
print(f"Placental Abruption cases: {len(df[df[outcomes[7]] == 1]):,}")
'''

# Saving Outcomes

In [ ]:
'''
print(f"\nTotal pregnancy episodes: {len(df):,}")
print(f"Total outcomes extracted: {len([col for col in df.columns if col in outcomes or '_12' in col or '_24' in col])}")

print(f"\nOutcome prevalence:")
for outcome in outcomes:
    if outcome in df.columns:
        n = df[outcome].sum()
        pct = n / len(df) * 100
        print(f"  {outcome}: {n:,} ({pct:.2f}%)")

print(f"\nFinal dataframe shape: {df.shape}")
'''

In [ ]:
'''
OUTPUT_FILE = 'pregnancy_cohort_outcomes_date.pkl'
df.to_pickle(OUTPUT_FILE)

print(f"\n Outcomes saved to: {OUTPUT_FILE}")
'''